In [ ]:
# =============================================================
# TAHAP 4: CASE SOLUTION REUSE
# Mata Kuliah Penalaran Komputer - CBR Korupsi Kerugian Keuangan Negara
# =============================================================

# Tahap 4: Case Solution Reuse
**Tujuan:** Menggunakan putusan kasus lama sebagai dasar rekomendasi
solusi untuk kasus baru melalui majority vote dan weighted similarity.

In [ ]:
Import Library
import os
import sys
import json
import pickle
import numpy as np
import pandas as pd
from collections import Counter
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings("ignore")

In [ ]:
Konfigurasi Path
BASE_DIR  = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) \
            if "__file__" in dir() else os.getcwd()
PROC_DIR  = os.path.join(BASE_DIR, "data", "processed")
EVAL_DIR  = os.path.join(BASE_DIR, "data", "eval")
RES_DIR   = os.path.join(BASE_DIR, "data", "results")
MODEL_DIR = os.path.join(BASE_DIR, "models")
RAW_DIR   = os.path.join(BASE_DIR, "data", "raw")
os.makedirs(RES_DIR, exist_ok=True)
print("✅ Path siap")

# Tambahkan path notebooks ke sys.path agar bisa import modul tahap sebelumnya
sys.path.insert(0, os.path.join(BASE_DIR, "notebooks"))

In [ ]:
Load Dataset & Model Retrieval
def load_all():
    """Load dataset dan model retrieval."""
    # Dataset
    csv_path = os.path.join(PROC_DIR, "cases_full.csv")
    if not os.path.exists(csv_path):
        csv_path = os.path.join(PROC_DIR, "cases.csv")
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    print(f"✅ Dataset: {len(df)} kasus")

    # Model TF-IDF
    vec_path = os.path.join(MODEL_DIR, "tfidf_vectorizer.pkl")
    mat_path = os.path.join(MODEL_DIR, "tfidf_matrix.npy")
    ids_path = os.path.join(MODEL_DIR, "case_ids.json")

    if not os.path.exists(vec_path):
        print("⚠️  Model belum ada. Jalankan Tahap 3 dulu!")
        return None, None, None, None

    with open(vec_path, "rb") as f:
        vectorizer = pickle.load(f)

    tfidf_matrix = np.load(mat_path)
    from scipy.sparse import csr_matrix
    tfidf_matrix = csr_matrix(tfidf_matrix)

    with open(ids_path, "r") as f:
        case_ids = json.load(f)

    print(f"✅ Model TF-IDF loaded | {len(case_ids)} vektor kasus")
    return df, vectorizer, tfidf_matrix, case_ids

In [ ]:
Bangun Case Solution Dictionary
def build_case_solutions(df: pd.DataFrame) -> Dict[str, Dict]:
    """
    Membangun dictionary solusi kasus:
    { case_id: { 'amar_putusan': ..., 'label': ..., 'pidana': ..., 'terdakwa': ... } }
    """
    solutions = {}
    for _, row in df.iterrows():
        solutions[row["case_id"]] = {
            "amar_putusan"  : str(row.get("amar_putusan",  "TIDAK_DITEMUKAN")),
            "label"         : str(row.get("label",          "bersalah")),
            "pidana_penjara": str(row.get("pidana_penjara", "TIDAK_DITEMUKAN")),
            "uang_pengganti": str(row.get("uang_pengganti", "TIDAK_DITEMUKAN")),
            "terdakwa"      : str(row.get("terdakwa",       "TIDAK_DITEMUKAN")),
            "pasal"         : str(row.get("pasal",          "TIDAK_DITEMUKAN")),
        }
    print(f"✅ {len(solutions)} solusi kasus siap")
    return solutions

In [ ]:
Fungsi retrieve() lokal (replikasi dari Tahap 3)
from sklearn.metrics.pairwise import cosine_similarity

def preprocess_text(text: str) -> str:
    import re
    stopwords_id = {
        "yang","dan","di","ke","dari","ini","itu","dengan","untuk","pada",
        "adalah","dalam","tidak","oleh","atau","juga","sudah","akan","ada",
        "bahwa","telah","tersebut","lebih","setelah","serta","dapat","harus",
        "namun","karena","atas","tapi","jika","maka","sebagai","antara","hal",
        "para","bagi","maupun","melalui","berdasarkan","terhadap","sehingga",
    }
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = [t for t in text.split() if t not in stopwords_id and len(t) > 2]
    return " ".join(tokens)

def retrieve(query: str, k: int = 5,
             vectorizer=None, tfidf_matrix=None, case_ids=None) -> List[Tuple]:
    """Fungsi retrieval: kembalikan top-k case_id beserta similarity score."""
    processed = preprocess_text(query)
    query_vec = vectorizer.transform([processed])
    sims      = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_idx   = sims.argsort()[::-1][:k]
    return [(case_ids[i], float(sims[i])) for i in top_idx]

In [ ]:
Algoritma Prediksi: Majority Vote
def majority_vote(solutions_list: List[Dict]) -> str:
    """Pilih label yang paling sering muncul di antara top-k kasus."""
    labels = [s["label"] for s in solutions_list]
    most_common = Counter(labels).most_common(1)[0][0]
    return most_common

In [ ]:
Algoritma Prediksi: Weighted Similarity
def weighted_similarity_vote(solutions_list: List[Dict],
                              scores: List[float]) -> str:
    """Pilih label berdasarkan bobot skor similarity (label dengan skor terbesar)."""
    weight_map = {}
    for sol, score in zip(solutions_list, scores):
        label = sol["label"]
        weight_map[label] = weight_map.get(label, 0) + score
    return max(weight_map, key=weight_map.get)

In [ ]:
Fungsi Utama: predict_outcome()
def predict_outcome(query: str, k: int = 5,
                    vectorizer=None, tfidf_matrix=None,
                    case_ids=None, case_solutions=None,
                    method: str = "weighted") -> Dict:
    """
    Prediksi solusi untuk kasus baru.
    Returns dict berisi prediksi label, amar gabungan, top-k cases, dll.
    """
    # Retrieval
    top_k_results = retrieve(query, k=k,
                              vectorizer=vectorizer,
                              tfidf_matrix=tfidf_matrix,
                              case_ids=case_ids)

    retrieved_ids = [r[0] for r in top_k_results]
    scores        = [r[1] for r in top_k_results]
    solutions     = [case_solutions[cid] for cid in retrieved_ids
                     if cid in case_solutions]

    if not solutions:
        return {"error": "Tidak ada kasus ditemukan"}

    # Prediksi label
    if method == "majority":
        predicted_label = majority_vote(solutions)
    else:
        predicted_label = weighted_similarity_vote(solutions, scores)

    # Ambil amar putusan dari kasus dengan similarity tertinggi sebagai referensi
    best_case_id = retrieved_ids[0]
    best_solution = case_solutions.get(best_case_id, {})
    best_amar = best_solution.get("amar_putusan", "")

    # Buat ringkasan prediksi
    pidana_list = [s["pidana_penjara"] for s in solutions
                   if s["pidana_penjara"] != "TIDAK_DITEMUKAN"]
    predicted_pidana = Counter(pidana_list).most_common(1)[0][0] if pidana_list else "N/A"

    return {
        "predicted_label"   : predicted_label,
        "predicted_pidana"  : predicted_pidana,
        "best_amar"         : best_amar[:500] if best_amar else "N/A",
        "top_k_case_ids"    : retrieved_ids,
        "top_k_scores"      : [round(s, 4) for s in scores],
        "method"            : method,
        "similarity_max"    : round(max(scores), 4),
        "similarity_avg"    : round(sum(scores)/len(scores), 4),
    }

In [ ]:
Demo Manual: 5 Kasus Baru
DEMO_QUERIES = [
    {
        "query_id"  : "demo_001",
        "query_text": (
            "terdakwa selaku kepala dinas pekerjaan umum melakukan korupsi "
            "dalam proyek pembangunan jalan senilai dua miliar rupiah dengan "
            "cara menggelembungkan anggaran dan meminta fee kepada kontraktor"
        ),
    },
    {
        "query_id"  : "demo_002",
        "query_text": (
            "terdakwa sebagai bendahara desa terbukti menyalahgunakan dana desa "
            "sebesar lima ratus juta rupiah untuk kepentingan pribadi dengan "
            "membuat laporan keuangan fiktif"
        ),
    },
    {
        "query_id"  : "demo_003",
        "query_text": (
            "terdakwa direktur bumn melakukan mark-up pengadaan barang dan jasa "
            "sehingga merugikan keuangan negara sebesar sepuluh miliar rupiah "
            "dengan modus pembuatan kontrak fiktif"
        ),
    },
    {
        "query_id"  : "demo_004",
        "query_text": (
            "terdakwa anggota dprd menerima suap dari pengusaha dalam proses "
            "pengesahan apbd sehingga negara dirugikan melalui proyek yang tidak "
            "sesuai spesifikasi teknis"
        ),
    },
    {
        "query_id"  : "demo_005",
        "query_text": (
            "terdakwa pejabat dinas kesehatan terbukti korupsi pengadaan alat "
            "kesehatan dengan cara menunjuk langsung rekanan tanpa tender dan "
            "menerima komisi sehingga merugikan negara tiga ratus juta rupiah"
        ),
    },
]

In [ ]:
Pipeline Utama
def run_solution_reuse_pipeline():
    print("\n" + "="*60)
    print("  TAHAP 4: CASE SOLUTION REUSE")
    print("="*60)

    # Load
    print("\n[1/4] Load dataset & model...")
    df, vectorizer, tfidf_matrix, case_ids = load_all()
    if df is None:
        return

    # Bangun solusi
    print("\n[2/4] Membangun case solutions dictionary...")
    case_solutions = build_case_solutions(df)

    # Simpan case_solutions
    with open(os.path.join(MODEL_DIR, "case_solutions.json"), "w", encoding="utf-8") as f:
        json.dump(case_solutions, f, ensure_ascii=False, indent=2)
    print("  Case solutions tersimpan.")

    # Demo prediksi
    print("\n[3/4] Demo prediksi 5 kasus baru...")
    print("-"*60)
    prediction_records = []

    for demo in DEMO_QUERIES:
        print(f"\n  🔍 {demo['query_id']}")
        print(f"  Query  : {demo['query_text'][:100]}...")

        result_mv = predict_outcome(
            demo["query_text"], k=5,
            vectorizer=vectorizer, tfidf_matrix=tfidf_matrix,
            case_ids=case_ids, case_solutions=case_solutions,
            method="majority"
        )
        result_ws = predict_outcome(
            demo["query_text"], k=5,
            vectorizer=vectorizer, tfidf_matrix=tfidf_matrix,
            case_ids=case_ids, case_solutions=case_solutions,
            method="weighted"
        )

        print(f"  Majority Vote    : {result_mv['predicted_label']} | pidana: {result_mv['predicted_pidana']}")
        print(f"  Weighted Sim.    : {result_ws['predicted_label']} | pidana: {result_ws['predicted_pidana']}")
        print(f"  Top-5 Cases      : {result_ws['top_k_case_ids']}")
        print(f"  Similarity Scores: {result_ws['top_k_scores']}")
        print(f"  Amar Referensi   : {result_ws['best_amar'][:150]}...")

        prediction_records.append({
            "query_id"            : demo["query_id"],
            "query_text"          : demo["query_text"][:200],
            "predicted_label_mv"  : result_mv["predicted_label"],
            "predicted_label_ws"  : result_ws["predicted_label"],
            "predicted_pidana"    : result_ws["predicted_pidana"],
            "top_5_case_ids"      : "|".join(result_ws["top_k_case_ids"]),
            "top_5_scores"        : "|".join(str(s) for s in result_ws["top_k_scores"]),
            "similarity_max"      : result_ws["similarity_max"],
            "similarity_avg"      : result_ws["similarity_avg"],
        })

    # Simpan predictions.csv
    print("\n[4/4] Menyimpan hasil prediksi...")
    pred_df  = pd.DataFrame(prediction_records)
    pred_path = os.path.join(RES_DIR, "predictions.csv")
    pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")
    print(f"  ✅ Prediksi tersimpan: {pred_path}")
    print("\n" + pred_df[["query_id","predicted_label_ws","predicted_pidana","similarity_max"]].to_string(index=False))
    print("\n✅ Tahap 4 selesai!")
    print("="*60)

    return case_solutions

if __name__ == "__main__":
    run_solution_reuse_pipeline()